## Numerical precision

### Goals

ONNX versus `sci-kit-learn` has known issues with different handling of [numerical precision](https://onnx.ai/sklearn-onnx/auto_tutorial/plot_ebegin_float_double.html). As the linked post describes, the effect of this is particularly problematic for tree-based models. This post shows an example.

### Set Up

In [21]:
import orbital
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification

In [ ]:
#| label: model-prep

# make data dataset
X_train_org, y_train = make_classification(10000, random_state = 102)
X_train_rnd = X_train_org.round(3).astype(np.float16)
X_train_int = (X_train_org*1000).round(0).astype(np.int64)

# get column names for use in pipeline
n_cols = len(X_train_org[0])
nm_cols = [f"f{i}" for i in range(n_cols)]

# fit sklearn pipeline
lbls = ['Double', 'Float', 'Int']
X = [X_train_org, 
     X_train_rnd, 
     X_train_int]
typ_orb = [orbital.types.DoubleColumnType(),
           orbital.types.FloatColumnType(),
           orbital.types.Int64ColumnType()]
ppls = []
sqls = []

for x, t in zip(X, typ_orb): 

  ppl = Pipeline([
  ("pre", ColumnTransformer([], remainder="passthrough")),
  ("gbm", GradientBoostingClassifier(max_depth = 10, n_estimators = 10, random_state = 504)),
    ])
  ppl.fit(x, y_train)
  ppls.append(ppl)

  feat_dict = {c:t for c in nm_cols}
  ppl_orb = orbital.parse_pipeline(ppl, features=feat_dict)
  sql = orbital.export_sql("DATA_TABLE", ppl_orb, dialect="duckdb")
  sqls.append(sql)

### Testing Output

We can now assess scoring time and double check the validity of our predictions.

In [ ]:
preds_ppl = []
preds_orb = []

for x, p, s in zip(X, ppls, sqls):
    
    DATA_TABLE = pd.DataFrame(x, columns = nm_cols)
    preds_orb.append( duckdb.sql(s).df()['output_probability.1'] )
    preds_ppl.append( p.predict_proba(x)[:,1] )


We can also confirm that our outputs still match in this more complex case. 

In [ ]:
#| label: valid-values

for lbl, ppl, orb in zip(lbls, preds_ppl, preds_orb):

    deltas = np.abs(ppl - orb)

    print(f"--- {lbl} ---")
    print(f"Pred (PPL): {ppl[0]:.4f}")
    print(f"Pred (ORB): {orb[0]:.4f}")
    print(f"ppl and orb match: {np.all(np.isclose(ppl, orb))}")
    print(f"ppl and orb prop mismatch: {sum(~np.isclose(ppl, orb)) / len(ppl)}")
    print(f"ppl and orb prop MAE: {sum(deltas) / len(ppl):.8f}")
    print(f"ppl and orb diff range: ({min(deltas):.6f}, {max(deltas):.6f})")
    print("\n")


In [25]:
#| label: model-prep

import warnings
warnings.filterwarnings('ignore')


types_or = [orbital.types.FloatColumnType(),
            orbital.types.Float16ColumnType(),
            orbital.types.DoubleColumnType(),
            orbital.types.Int64ColumnType(),
            orbital.types.UInt64ColumnType(),
            orbital.types.Int32ColumnType(),
            orbital.types.UInt32ColumnType(),
            orbital.types.Int16ColumnType(),
            orbital.types.UInt16ColumnType(),
            orbital.types.Int8ColumnType(),
            orbital.types.UInt8ColumnType(),
            orbital.types.BooleanColumnType()]
types_np = [np.float32,
            np.float16, 
            np.float64, 
            np.int64, 
            np.uint64, 
            np.int32, 
            np.uint32, 
            np.int16, 
            np.uint16, 
            np.int8, 
            np.uint8, 
            np.bool]

# make data dataset
X_train, y_train = make_classification(10000, random_state = 102)

# get column names for use in pipeline
n_cols = len(X_train[0])
nm_cols = [f"f{i}" for i in range(n_cols)]

try:
  print("Here's variable x:", x)
except Exception as error:
  print("An error occurred:", error)

# iterate on types
for n, o in zip(types_np, types_or):

    try:

      X_train_prep = X_train.astype(n)
      feat_dict = {c:o for c in nm_cols}

      ppl = Pipeline([
      ("pre", ColumnTransformer([], remainder="passthrough")),
      ("gbm", RandomForestClassifier(max_depth = 1, n_estimators = 1, random_state = 504)),
      ])
      ppl.fit(X_train_prep, y_train)    
      orb = orbital.parse_pipeline(ppl, features=feat_dict)
      sql = orbital.export_sql("DATA_TABLE", orb, dialect="duckdb")

      print(f"{o} works")
    
    except Exception as error:

      print(f"{o} fails: {error}")

An error occurred: name 'x' is not defined
FloatColumnType() works
Float16ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.Float16TensorType'>'.
DoubleColumnType() works
An error occurred: name 'x' is not defined
FloatColumnType() works
Float16ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.Float16TensorType'>'.
DoubleColumnType() works


Int64ColumnType() works
UInt64ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.UInt64TensorType'>'.
Int32ColumnType() fails: Operator SklearnRandomForestClassifier (type: SklearnRandomForestClassifier) got an input variable with a wrong type <class 'skl2onnx.common.data_types.Int32TensorType'>. Only [<class 'skl2onnx.common.data_types.BooleanTensorType'>, <class 'skl2onnx.common.data_types.DoubleTensorType'>, <class 'skl2onnx.common.data_types.FloatTensorType'>, <class 'skl2onnx.common.data_types.Int64TensorType'>] are allowed
UInt32ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.UInt32TensorType'>'.
Int16ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.Int16TensorType'>'.
UInt16ColumnType() fails: data_type is not a tensor type but '<class 'skl2onnx.common.data_types.UInt16TensorType'>'.
Int8ColumnType() fails: Operator SklearnRandomForestClassifier (type: Skl